# Advanced RAG: BM25+Dense Hybrid 검색 + Cross-Encoder Reranker

1. BM25(키워드 기반) 검색기 구성
2. Dense(임베딩 기반) 검색기 구성
3. EnsembleRetriever로 Hybrid 검색 결합 (RRF)
4. Cross-Encoder Reranker 적용
5. Dense only / Hybrid / Hybrid+Reranker 3조건 비교 실험


## 0. 환경 설정 및 패키지 설치

In [1]:
!pip install -q rank_bm25 langchain langchain_core langchain-community langchain-huggingface sentence-transformers chromadb konlpy


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 86.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 46.2 MB/s eta 0:00:00


In [2]:
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_structured.json
!wget https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json

--2026-06-25 13:30:03--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports_structured.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 140040 (137K) [text/plain]
Saving to: ‘reports_structured.json’

reports_structured. 100%[===================>] 136.76K  --.-KB/s    in 0.002s  

2026-06-25 13:30:03 (69.8 MB/s) - ‘reports_structured.json’ saved [140040/140040]

--2026-06-25 13:30:03--  https://raw.githubusercontent.com/ws-l/hh_2026_5/main/data/reports.json
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 126861

In [3]:
import json
import time
import numpy as np
import pandas as pd

with open("reports.json", "r", encoding="utf-8") as f:
    reports = json.load(f)

documents_text = [r["report_text"] for r in reports]
report_ids = [r["report_id"] for r in reports]

print(f"총 {len(reports)}건 로드")


총 150건 로드


## 1. BM25 검색기 구성

BM25는 형태소 단위 토큰화 품질이 검색 정확도에 영향: 한국어는 공백 기준 분리만으로는 부족해 형태소 분석기(Okt)를 사용할 수 있음


In [4]:
from konlpy.tag import Okt
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

okt = Okt()

def korean_tokenizer(text):
    return okt.morphs(text)

lc_documents = [
    Document(page_content=text, metadata={"report_id": rid})
    for text, rid in zip(documents_text, report_ids)
]

bm25_retriever = BM25Retriever.from_documents(
    lc_documents,
    preprocess_func=korean_tokenizer,
)
bm25_retriever.k = 5

print("BM25 검색기 구성 완료")


/tmp/ipykernel_28451/2798951340.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import BM25Retriever


BM25 검색기 구성 완료


In [5]:
# BM25 검색 테스트 - 정확한 키워드(설비명)가 포함된 질의에 강점을 보이는지 확인
query_keyword = "CNC 가공기 실린더블록용 A-3호기"
bm25_results = bm25_retriever.invoke(query_keyword)

print(f"질의: {query_keyword}\n")
for doc in bm25_results[:3]:
    print(f"[{doc.metadata['report_id']}] {doc.page_content[:100]}")


질의: CNC 가공기 실린더블록용 A-3호기

[RPT-121] 2024년 12월 1일 08:40, CNC머신 A-1호기에서 정밀 하우징의 기준 홀 위치가 도면 좌표 대비 X방향 +0.42mm, Y방향 -0.31mm 이탈한 것을 확인함. 동일 
[RPT-005] 2024년 1월 12일 08:50, CNC 머신 A-3호기에서 기어 하우징 포켓 깊이가 도면 8.50mm 대비 8.07~8.12mm로 얕게 가공된 사실을 확인함. Z축 원점 보정값
[RPT-022] 2024년 3월 5일 13:35, CNC 머신 F-2호기에서 가이드 레일 고정 홀 간 거리가 기준 75.00mm 대비 75.26~75.33mm로 크게 가공됨을 확인함. Y축 위치 


## 2. Dense 검색기 구성 


In [6]:
from langchain_community.vectorstores import Chroma as LC_Chroma
from langchain_huggingface import HuggingFaceEmbeddings

lc_embeddings = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

dense_vectorstore = LC_Chroma.from_documents(
    documents=lc_documents,
    embedding=lc_embeddings,
    collection_name="dense_hybrid_reports",
)
dense_retriever = dense_vectorstore.as_retriever(search_kwargs={"k": 5})

print("Dense 검색기 구성 완료")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:134: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Dense 검색기 구성 완료


In [ ]:
# Dense 검색 테스트 - 서술형(증상 묘사) 질의에 대해 확인
query_semantic = "진동이 심하고 소음이 발생하는 상황"
dense_results = dense_retriever.invoke(query_semantic)

print(f"질의: {query_semantic}\n")
for doc in dense_results[:3]:
    print(f"[{doc.metadata['report_id']}] {doc.page_content[:100]}")


질의: 진동이 심하고 소음이 발생하는 상황

[RPT-052] 2024년 6월 8일 16:35, CO2 용접기 #7호기에서 용접 완료된 케이스 모서리 접합부의 헬륨 누설 시험 결과 1.8×10^-3 mbar·L/s가 검출됨. 용접 토치 팁 마
[RPT-081] 2024년 9월 2일 08:45, 자동포장기 C-2호기에서 알루미늄 라미네이트 포장재 실링부에 짙은 회색 변색이 발생함을 확인함. 실링압력은 5.8bar로 높았고 압력 상한 기준이
[RPT-016] 2024년 2월 16일 09:00, 사출성형기 #7호기에서 POM 레버의 전체 길이가 기준 86.00mm 대비 86.22~86.31mm로 길게 측정됨을 확인함. 사출압력은 178M


## 3. Hybrid Search: EnsembleRetriever (BM25 + Dense, RRF 결합)


In [8]:
from langchain_classic.retrievers import EnsembleRetriever   # ← langchain.retrievers 가 아님

# 가중치는 도메인 특성에 따라 튜닝 - 여기선 균등 시작
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],  # BM25:Dense = 4:6
)

print("Hybrid 검색기(EnsembleRetriever) 구성 완료")


Hybrid 검색기(EnsembleRetriever) 구성 완료


In [9]:
# 키워드+서술이 섞인 질의로 테스트
query_hybrid = "실린더블록 가공 중 진동이나 소음이 발생한 사례"
hybrid_results = hybrid_retriever.invoke(query_hybrid)

print(f"질의: {query_hybrid}\n")
for doc in hybrid_results[:5]:
    print(f"[{doc.metadata['report_id']}] {doc.page_content[:100]}")


질의: 실린더블록 가공 중 진동이나 소음이 발생한 사례

[RPT-007] 2024년 1월 18일 11:05, CNC 머신 B-2호기에서 가공품 측면에 폭 0.15~0.22mm의 선형 스크래치가 반복 발생함을 확인함. 칩 배출 컨베이어 지연으로 절삭 칩이
[RPT-020] 2024년 2월 28일 16:50, CNC 머신 F-1호기에서 알루미늄 판재 상면에 원형 공구 자국과 미세 스크래치가 혼재되어 발생함을 확인함. 절삭유 압력은 정상 0.35MPa 
[RPT-052] 2024년 6월 8일 16:35, CO2 용접기 #7호기에서 용접 완료된 케이스 모서리 접합부의 헬륨 누설 시험 결과 1.8×10^-3 mbar·L/s가 검출됨. 용접 토치 팁 마
[RPT-002] 2024년 1월 5일 14:30, CNC 머신 A-2호기에서 스테인리스 커버 외곽 절삭면에 길이 18~25mm의 미세 스크래치가 관찰됨. 절삭온도는 165°C까지 상승했고 공구 측
[RPT-025] 2024년 3월 14일 11:10, 사출성형기 #11호기에서 PA 하우징 표면 웰드라인 주변에 3~5mm 길이의 미세 크랙이 발생함을 확인함. 사출압력은 82MPa로 낮았고 실린더


- 설비명·부품번호가 포함된 질의는 BM25 비중이 높을 때 더 정확하게 검색됨
- 증상을 서술한 질의는 Dense 비중이 높을 때 의미적으로 더 가까운 문서를 찾음
- `weights` 값을 바꿔가며(예: [0.7, 0.3] vs [0.3, 0.7]) 결과가 어떻게 달라지는지 관찰


## 4. Cross-Encoder Reranker 적용

1차로 Hybrid 검색에서 더 넓게 후보를 가져온 뒤(Top-10~20), Cross-Encoder로 재정렬해 최종 Top-K만 찾음


In [10]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder("BAAI/bge-reranker-base")
print("Cross-Encoder Reranker 로드 완료")


config.json:   0%|          | 0.00/799 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Cross-Encoder Reranker 로드 완료


In [11]:
def rerank(query, documents, top_k=3):
    pairs = [[query, doc.page_content] for doc in documents]
    scores = reranker_model.predict(pairs)
    scored_docs = list(zip(documents, scores))
    scored_docs.sort(key=lambda x: x[1], reverse=True)
    return scored_docs[:top_k]

# 1차 Hybrid 검색으로 넓게 후보 확보 (Top-10)
hybrid_retriever_wide = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],
)
bm25_retriever.k = 10  # 1차 후보 폭 확대

query = "실린더블록 가공 중 진동이나 소음이 발생한 사례"
wide_candidates = hybrid_retriever_wide.invoke(query)
print(f"1차 후보 수: {len(wide_candidates)}")

reranked = rerank(query, wide_candidates, top_k=3)

print("\n=== Reranker 적용 후 최종 Top-3 ===")
for doc, score in reranked:
    print(f"score={score:.4f} [{doc.metadata['report_id']}] {doc.page_content[:100]}")


1차 후보 수: 14

=== Reranker 적용 후 최종 Top-3 ===
score=0.0429 [RPT-002] 2024년 1월 5일 14:30, CNC 머신 A-2호기에서 스테인리스 커버 외곽 절삭면에 길이 18~25mm의 미세 스크래치가 관찰됨. 절삭온도는 165°C까지 상승했고 공구 측
score=0.0020 [RPT-007] 2024년 1월 18일 11:05, CNC 머신 B-2호기에서 가공품 측면에 폭 0.15~0.22mm의 선형 스크래치가 반복 발생함을 확인함. 칩 배출 컨베이어 지연으로 절삭 칩이
score=0.0009 [RPT-123] 2024년 12월 3일 09:30, CNC머신 B-1호기에서 가공 후 조립된 브래킷의 체결 볼트 4개 중 2개가 지정 토크 18N·m 대비 7~9N·m 수준으로 체결된 것을 확인함


**관찰 포인트**
- Reranker 적용 전(Hybrid 1차 순위)과 적용 후(Cross-Encoder 재정렬) 순위가 바뀌는 문서가 있는지 확인
- Cross-Encoder는 질의-문서 쌍을 함께 인코딩하므로 Bi-encoder(Dense 검색)보다 느리지만 더 정밀함


## 5. 3조건 비교 실험: Dense only / Hybrid / Hybrid+Reranker

동일 질의 세트로 세 가지 검색 조건의 결과를 나란히 비교합니다.


In [12]:
test_queries = [
    "CNC 가공기 실린더블록용 A-3호기에서 발생한 불량",          # 키워드 위주
    "표면이 거칠고 광택이 일정하지 않은 증상",                   # 서술형 위주
    "열처리 공정 후 경도가 기준치에 미달하는 경우",              # 혼합형
    "조립 과정에서 체결이 헐겁거나 토크가 부족한 사례",          # 혼합형
]

def get_top_ids(retriever, query, k=3):
    docs = retriever.invoke(query)[:k]
    return [d.metadata["report_id"] for d in docs]

def get_hybrid_reranked_ids(query, k=3):
    bm25_retriever.k = 10
    candidates = hybrid_retriever_wide.invoke(query)
    reranked_docs = rerank(query, candidates, top_k=k)
    return [d.metadata["report_id"] for d, _ in reranked_docs]

comparison_rows = []
for q in test_queries:
    dense_ids = get_top_ids(dense_retriever, q, k=3)
    bm25_retriever.k = 5
    hybrid_ids = get_top_ids(hybrid_retriever, q, k=3)
    reranked_ids = get_hybrid_reranked_ids(q, k=3)
    comparison_rows.append({
        "질의": q,
        "Dense only": dense_ids,
        "Hybrid": hybrid_ids,
        "Hybrid+Reranker": reranked_ids,
    })

comparison_df = pd.DataFrame(comparison_rows)
pd.set_option("display.max_colwidth", None)
comparison_df


,질의,Dense only,Hybrid,Hybrid+Reranker
0,CNC 가공기 실린더블록용 A-3호기에서 발생한 불량,"[RPT-027, RPT-024, RPT-007]","[RPT-027, RPT-024, RPT-007]","[RPT-005, RPT-010, RPT-011]"
1,표면이 거칠고 광택이 일정하지 않은 증상,"[RPT-081, RPT-063, RPT-071]","[RPT-081, RPT-063, RPT-071]","[RPT-081, RPT-071, RPT-054]"
2,열처리 공정 후 경도가 기준치에 미달하는 경우,"[RPT-094, RPT-125, RPT-025]","[RPT-094, RPT-125, RPT-025]","[RPT-025, RPT-126, RPT-038]"
3,조립 과정에서 체결이 헐겁거나 토크가 부족한 사례,"[RPT-003, RPT-093, RPT-017]","[RPT-003, RPT-093, RPT-017]","[RPT-017, RPT-003, RPT-093]"


In [13]:
# 질의별로 Top-3 결과가 조건마다 어떻게 달라지는지 상세 출력
for row in comparison_rows:
    print(f"\n질의: {row['질의']}")
    print(f"  Dense only       : {row['Dense only']}")
    print(f"  Hybrid           : {row['Hybrid']}")
    print(f"  Hybrid+Reranker  : {row['Hybrid+Reranker']}")



질의: CNC 가공기 실린더블록용 A-3호기에서 발생한 불량
  Dense only       : ['RPT-027', 'RPT-024', 'RPT-007']
  Hybrid           : ['RPT-027', 'RPT-024', 'RPT-007']
  Hybrid+Reranker  : ['RPT-005', 'RPT-010', 'RPT-011']

질의: 표면이 거칠고 광택이 일정하지 않은 증상
  Dense only       : ['RPT-081', 'RPT-063', 'RPT-071']
  Hybrid           : ['RPT-081', 'RPT-063', 'RPT-071']
  Hybrid+Reranker  : ['RPT-081', 'RPT-071', 'RPT-054']

질의: 열처리 공정 후 경도가 기준치에 미달하는 경우
  Dense only       : ['RPT-094', 'RPT-125', 'RPT-025']
  Hybrid           : ['RPT-094', 'RPT-125', 'RPT-025']
  Hybrid+Reranker  : ['RPT-025', 'RPT-126', 'RPT-038']

질의: 조립 과정에서 체결이 헐겁거나 토크가 부족한 사례
  Dense only       : ['RPT-003', 'RPT-093', 'RPT-017']
  Hybrid           : ['RPT-003', 'RPT-093', 'RPT-017']
  Hybrid+Reranker  : ['RPT-017', 'RPT-003', 'RPT-093']


- 키워드 위주 질의에서는 Dense only → Hybrid 전환 시 결과가 크게 개선되는 경향
- 서술형 질의에서는 Hybrid → Hybrid+Reranker 전환 시 상위 문서의 관련성이 더 정밀해지는 경향
- 모든 질의에서 개선되지 않으며, 어떤 질의 유형에서 차이가 작거나 없는지도 확인
- Hybrid+Reranker 파이프라인이 다음의 GraphRAG와 비교될 "Advanced 벡터RAG" 기준선


# 6. 카드VOC로 위의 작업을 수행해보세요.